# RF-DETR on NFL Helmets (Single View)

The RF-DETR model is a state-of-the-art object detection model that has shown impressive performance on common object detection benchmarks, such as COCO. This performance rivals the YOLO group of models.

We will use this model to attempt to finetune a RF-DETR model on existing NFL helmets. The data we currently have is in video format, in two different views. To keep things simple, we will focus on the **endzone** view when detecting helmets. The expected outputs are bounding boxes corresponding to all the helmets in the image.

We have two distinct datasets to work with. One is the set of continuous videos, with some labeled data, although not for every frame in the videos. Then we have also images, for which bounding box labels are given.

**We will start with still images since they are easier to preprocess.**

## Import Packages

In [ ]:
# From RF-DETR developers (Roboflow)
from rfdetr import RFDETRMedium
import supervision as sv

import numpy as np
import polars
import os
import glob
import datetime as dt
import shutil
import itertools
import copy
import json

import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from tqdm.notebook import tqdm
import cv2

RANDOM_SEED = 1729
TEST_SIZE = 0.2

DATA_DIR = os.path.join('../data', 'nfl-health-and-safety-helmet-assignment')
DATASET_DIR = os.path.join(DATA_DIR, 'nfl_helmet_image_dataset')


## Training on Still Images
Before we jump into the videos, we will train on the provided ~10000 images. But before we use the images directly, we need to place them in the right format. This includes structuring our directories correctly, and also creating the `annotations.json` file that is common with the COCO datasets.

Originally, the images are located in the top-level directory with `./data/nfl-health-and-safety-helmet-assignment`. We will create another folder here and establish the format.

```text
dataset/
├── train/
│   ├── _annotations.coco.json
│   ├── image1.jpg
│   ├── image2.jpg
│   └── ... (other image files)
├── valid/
│   ├── _annotations.coco.json
│   ├── image1.jpg
│   ├── image2.jpg
│   └── ... (other image files)
└── test/
    ├── _annotations.coco.json
    ├── image1.jpg
    ├── image2.jpg
    └── ... (other image files)
```

### Putting Images into Structure
We will follow the structure above and **move** the files (so we don't take up double the storage space). We will also create two separate datasets for the endzone and sideline views, since they are fundamentally different views (although they are of the same scene).

In [ ]:
ENDZONE_DIR = os.path.join(DATASET_DIR, 'Endzone')
SIDELINE_DIR = os.path.join(DATASET_DIR, 'Sideline')
os.makedirs(ENDZONE_DIR, exist_ok=True)
os.makedirs(SIDELINE_DIR, exist_ok=True)
# Create the directories. We will just have a train and test datasets.
# First find the split...
for view, directory in zip(['Endzone', 'Sideline'], [ENDZONE_DIR, SIDELINE_DIR]):
    print(f'Gathering the {view} view...')
    all_og_image_files = glob.glob(os.path.join(DATA_DIR, 'images', f'*{view}*.jpg'))
    # Create a train test split. Make it reproducible...
    train, test = train_test_split(all_og_image_files, test_size=TEST_SIZE, random_state=RANDOM_SEED)
    valid, test = train_test_split(test, test_size=0.5, random_state=RANDOM_SEED)
    print('Training images:', len(train))
    print('Validation images:', len(valid))
    print('Test images:', len(test))
    # Create the train and test directories inside...
    os.makedirs(os.path.join(directory, 'train'), exist_ok=True)
    os.makedirs(os.path.join(directory, 'valid'), exist_ok=True)
    os.makedirs(os.path.join(directory, 'test'), exist_ok=True)
    # Move the files to the sub-directory (/train or /test)
    for dirname, dataset in zip(['train', 'valid', 'test'], [train, valid, test]):
        for file in tqdm(dataset, desc=f'Moving {dirname} files'):
            shutil.move(file, os.path.join(directory, dirname, os.path.basename(file)))
    print()


### Creating Annotations
Now that the images are in the right structure, we need to create an annotations file. This is the file that essentially contains the labels for the bounding boxes in each image (remember there will be multiple helmets in each image). [Here is what the annotations JSON should look like](https://roboflow.com/formats/coco-json).

For the purposes of this project, we just have one kind of annotation, the helmet. In our ground truth labeling, we have `Helmet`, `Helmet-Blurred`, `Helmet-Difficult`, and `Helmet-Sideline`. We will consolidate all of these into a single `Helmet` label.

In [ ]:
image_labels = polars.read_csv(os.path.join(DATA_DIR, 'image_labels.csv'))
print(image_labels.head())
print(image_labels.shape)

In [ ]:
# Add columns that tell us what view it is, and whether it falls into the train or test set.
structured_files = glob.glob(os.path.join(DATASET_DIR, '**/*.jpg'), recursive=True)
filenames = [os.path.basename(file) for file in structured_files]
dirnames = [os.path.basename(os.path.dirname(file)) for file in structured_files]
get_train_test = lambda filename: dirnames[filenames.index(os.path.basename(filename))]
image_labels = image_labels.with_columns(
    polars.when(polars.col('image').str.contains('Endzone'))
        .then(polars.lit('Endzone'))
        .otherwise(polars.lit('Sideline'))
        .alias('view'),
    polars.col('image').map_elements(get_train_test, return_dtype=polars.String).alias('split')
)
print(image_labels.head())

'We are ready to do the annotations. The following in a sample of what the file should look like.

```json
{
    "info": {
        "year": "2020",
        "version": "1",
        "description": "Exported from roboflow.ai",
        "contributor": "Roboflow",
        "url": "https://app.roboflow.ai/datasets/hard-hat-sample/1",
        "date_created": "2000-01-01T00:00:00+00:00"
    },
    "licenses": [
        {
            "id": 1,
            "url": "https://creativecommons.org/publicdomain/zero/1.0/",
            "name": "Public Domain"
        }
    ],
    "categories": [
        {
            "id": 0,
            "name": "Workers",
            "supercategory": "none"
        },
        {
            "id": 1,
            "name": "head",
            "supercategory": "Workers"
        },
        {
            "id": 2,
            "name": "helmet",
            "supercategory": "Workers"
        },
        {
            "id": 3,
            "name": "person",
            "supercategory": "Workers"
        }
    ],
    "images": [
        {
            "id": 0,
            "license": 1,
            "file_name": "0001.jpg",
            "height": 275,
            "width": 490,
            "date_captured": "2020-07-20T19:39:26+00:00"
        }
    ],
    "annotations": [
        {
            "id": 0,
            "image_id": 0,
            "category_id": 2,
            "bbox": [
                45,
                2,
                85,
                85
            ],
            "area": 7225,
            "segmentation": [],
            "iscrowd": 0
        },
        {
            "id": 1,
            "image_id": 0,
            "category_id": 2,
            "bbox": [
                324,
                29,
                72,
                81
            ],
            "area": 5832,
            "segmentation": [],
            "iscrowd": 0
        }
    ]
}
```

Some notes:
- One file should be generated for each of the train/test split directories, and it should only reference the files in that directory.
- We only have helmets to worry about, so the `"category"` will only have one entry in it.
- Other areas such as `"date_captured"` and `"segmentation"` can be whatever we want.

In [ ]:
# There is some info that will be the same across the 4 files
base_annotations = {
    "info": {
        "year": "2025",
        "version": "1",
        "description": "Generated COCO annotations for NFL Helmet detection",
        "contributor": "mughil",
        "url": "",
        "date_created": "2025-10-10T00:00:00+00:00"
    },
    "licenses": [
        {
            "id": 1,
            "url": "https://creativecommons.org/publicdomain/zero/1.0/",
            "name": "Public Domain"
        }
    ],
    "categories": [
        {
            "id": 0,
            "name": "Helmet",
            "supercategory": "none"
        }
    ]
}

Each row of `image_labels` corresponds to one bounding box in the specified image. They are also marked on if the image is the endzone/sideline view, and also if it's part of the train/test set.

This will cleanly go into how we add to the `images` and `annotations` arrays. The end result is the creation of 4 separate annotations JSONs. Unfortunately, in `polars` it's generally discouraged to iterate through the rows directly. But there are no other ways.

In [ ]:
for view, split in itertools.product(['Endzone', 'Sideline'], ['train', 'valid', 'test']):
    # Copy the base annotations...
    annotations = copy.copy(base_annotations)
    # Filter the dataframe according to the view and split,
    # and SORT according to the image file name.
    # This will come in handy later.
    dataset_view_split = image_labels.filter(
        polars.col('view').eq(view),
        polars.col('split').eq(split)
    ).sort('image', descending=False)
    # Create separate arrays for the images
    images = []
    bounding_boxes = []
    # For each row, keep track of the current image and its ID. As soon as
    # the image changes, update the ID, read it, and grab the
    # width and height...
    image = dataset_view_split[0, 'image']
    image_obj = cv2.imread(os.path.join(DATASET_DIR, view, split, image))
    width, height = image_obj.shape[:2]
    image_id = 0
    images.append({
        'id': image_id,
        'license': 1,
        'file_name': image,
        'width': width,
        'height': height,
        'date_captured': dt.datetime.now(dt.timezone.utc).strftime('%Y-%m-%dT%H:%M:%SZ'),
    })
    # Iterate through rows, keeping track of the box IDs
    for box_id, row in tqdm(enumerate(dataset_view_split.iter_rows(named=True)),
                            desc=f'Generating annotations JSON for {view} ({split})',
                            total=dataset_view_split.shape[0]):
        # If we encounter a new image, then read it in, and append it to the image array.
        if image != row['image']:
            image = row['image']
            image_obj = cv2.imread(os.path.join(DATASET_DIR, view, split, image))
            width, height = image_obj.shape[:2]
            image_id += 1
            images.append({
                'id': image_id,
                'license': 1,
                'file_name': image,
                'width': width,
                'height': height,
                'date_captured': dt.datetime.now(dt.timezone.utc).strftime('%Y-%m-%dT%H:%M:%SZ')
            })
        # Get the coordinates
        x, y, width, height = row['left'], row['top'], row['width'], row['height']
        # Add a JSON dictionary to the bounding box array
        bounding_boxes.append({
            'id': box_id,
            'image_id': image_id,
            'category_id': 0,  # We only have one category
            'bbox': [x, y, width, height],
            'area': width * height,
            'segmentation': [],
            'iscrowd': 0,
        })
    # Both dictionaries are done. Add to the larger annotations JSON,
    # and save the file in the corresponding directory
    annotations['images'] = images
    annotations['annotations'] = bounding_boxes

    with open(os.path.join(DATASET_DIR, view, split, '_annotations.coco.json'), 'w') as f:
        json.dump(annotations, f, indent=4)


### Validating and Visualizing
Now that the conversion has finished, we will plot a sample of images to make sure everything worked as intended. To easily plot them, we will use the `supervision` package and its built-in plotting capabilities.

In [ ]:
N_SAMPLE = 2
for view, split in itertools.product(['Endzone', 'Sideline'], ['train', 'valid', 'test']):
    print(f'Plotting a sample of {N_SAMPLE} samples from {view} ({split})')
    with open(os.path.join(DATASET_DIR, view, split, '_annotations.coco.json'), 'r') as f:
        annotations = json.load(f)
    samples = np.random.choice(annotations['images'], size=N_SAMPLE, replace=False)
    for sample in samples:
        # Extract the image ID, and its corresponding annotations
        image_id = sample['id']
        image_annotations = np.asarray([box_annotation['bbox'] for box_annotation in annotations['annotations']
                                            if box_annotation['image_id'] == image_id])
        # We need xyxy, not xy width height
        if len(image_annotations) != 0:
            image_annotations[:, 2] += image_annotations[:, 0]
            image_annotations[:, 3] += image_annotations[:, 1]
        # Read the image and create a Detections object and plot with the boxes overlaid
        image = cv2.imread(os.path.join(DATASET_DIR, view, split, sample['file_name']))
        detections = sv.Detections(
            image_annotations,
            class_id=np.zeros(image_annotations.shape[0], dtype=int) + 4,  # Just controlling the color of the box
        )
        box_annotator = sv.BoxAnnotator()
        annotated_frame = box_annotator.annotate(
            scene=image.copy(),
            detections=detections
        )
        sv.plot_image(annotated_frame)
    print()



## Train model
Everything has been set up to train a sample RF-DETR

In [ ]:
from rfdetr import RFDETRBase
import torch
import pytorch_lightning as pl
from torch.utils.data import DataLoader
from ultralytics.data import build

In [ ]:
model = RFDETRBase()
OUTPUT_PATH = os.path.join(DATASET_DIR, 'outputs')
os.makedirs(OUTPUT_PATH, exist_ok=True)

In [ ]:
model.train(
    dataset_dir=os.path.join(DATASET_DIR, 'Endzone'),
    epochs=5,
    batch_size=4,
    grad_accum_steps=4,
    lr=1e-4,
    output_dir=OUTPUT_PATH,
    device='mps'
)